# Multi-Step Pipelines

Pipelines let you chain multiple `dataset-builder` commands into a single automated workflow.

Instead of running commands one by one, you define a **pipeline JSON config file** that lists each step in order. The `run` command then executes them sequentially, passing outputs from one step as inputs to the next.

This is useful for:
- **Reproducible workflows** — save your exact processing steps as a config file
- **Batch processing** — run complex multi-step jobs with a single command
- **Automation** — integrate into CI/CD or scheduled tasks

In [ ]:
import json
from pathlib import Path

output_dir = Path("../output")
output_dir.mkdir(exist_ok=True)

simple_pipeline = {
    "name": "Simple Pipeline",
    "steps": [
        {
            "command": "generate",
            "args": {
                "type": "person",
                "count": 10,
                "output": "../output/nb_pipe_people.json"
            }
        },
        {
            "command": "export",
            "args": {
                "input": "../output/nb_pipe_people.json",
                "output": "../output/nb_pipe_people.csv",
                "format": "csv"
            }
        }
    ]
}

pipeline_path = output_dir / "nb_pipeline_simple.json"
with open(pipeline_path, "w", encoding="utf-8") as f:
    json.dump(simple_pipeline, f, indent=2)

print(f"Pipeline config written to: {pipeline_path}")
print(json.dumps(simple_pipeline, indent=2))

In [ ]:
!npm start -- run -c ../output/nb_pipeline_simple.json --verbose

In [ ]:
import json
import csv
from pathlib import Path

# Check that output files were created
json_path = Path("../output/nb_pipe_people.json")
csv_path = Path("../output/nb_pipe_people.csv")

print(f"JSON output exists: {json_path.exists()}")
print(f"CSV output exists:  {csv_path.exists()}")
print()

if json_path.exists():
    data = json.loads(json_path.read_text(encoding="utf-8"))
    print(f"Generated {len(data)} person records")
    print(f"Fields: {list(data[0].keys()) if data else 'N/A'}")
    print()
    print("First 3 records:")
    for i, rec in enumerate(data[:3]):
        print(f"  {i+1}. {json.dumps(rec)}")

if csv_path.exists():
    print()
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    print(f"CSV has {len(rows)} rows with columns: {list(rows[0].keys()) if rows else 'N/A'}")

## Pipeline Step Types Reference

Any `dataset-builder` command can be used as a pipeline step. Here's a reference of available commands:

| Command | Description | Key Args |
|---------|-------------|----------|
| **`scrape`** | Scrape data from websites | `url`, `query`, `engine`, `output` |
| **`generate`** | Generate synthetic data (Faker or LLM) | `type`, `count`, `output`, `model` |
| **`import`** | Import from CSV, JSON, XML, XLS files | `file`, `output` |
| **`export`** | Export to various file formats | `input`, `output`, `format` |
| **`transform`** | Transform scraped data into structured records | `input`, `output`, `model` |
| **`format`** | Format for ML frameworks (Alpaca, ChatML, etc.) | `input`, `output`, `format` |
| **`clean`** | Clean data (dedupe, trim, remove empty) | `input`, `output`, `dedupe`, `trim` |
| **`translate`** | Translate datasets via LLM | `input`, `output`, `language`, `model` |
| **`compress`** | Compress media assets (ffmpeg) | `input`, `output`, `quality` |

Each step in the pipeline config uses the format:
```json
{"command": "<command>", "args": {"<key>": "<value>", ...}}
```

In [ ]:
import json
from pathlib import Path

complex_pipeline = {
    "name": "Generate, Clean, and Format Pipeline",
    "steps": [
        {
            "command": "generate",
            "args": {
                "type": "qa",
                "count": 20,
                "output": "../output/nb_pipe_qa_raw.json"
            }
        },
        {
            "command": "clean",
            "args": {
                "input": "../output/nb_pipe_qa_raw.json",
                "output": "../output/nb_pipe_qa_clean.json",
                "dedupe": true,
                "trim": true,
                "remove-empty": true
            }
        },
        {
            "command": "format",
            "args": {
                "input": "../output/nb_pipe_qa_clean.json",
                "output": "../output/nb_pipe_qa_alpaca/",
                "format": "alpaca"
            }
        }
    ]
}

pipeline_path = Path("../output/nb_pipeline_complex.json")
with open(pipeline_path, "w", encoding="utf-8") as f:
    json.dump(complex_pipeline, f, indent=2)

print(f"Complex pipeline config written to: {pipeline_path}")
print(json.dumps(complex_pipeline, indent=2))

In [ ]:
!npm start -- run -c ../output/nb_pipeline_complex.json --verbose

## Tips for Pipeline Workflows

### Error Handling
- If a step fails, the pipeline stops at that step and reports the error
- Use `--verbose` to see detailed output from each step for debugging
- Check intermediate output files to pinpoint where things went wrong

### Verbose Mode
- Always use `--verbose` during development to see what each step is doing
- Once your pipeline is working, you can drop the flag for cleaner output

### Checking Intermediate Outputs
- Give each step a unique output path so you can inspect results at each stage
- Use Python cells between pipeline runs to verify data quality
- Keep intermediate files during development, clean up when done

### Best Practices
- Start with a simple pipeline and add steps incrementally
- Use descriptive `name` fields in your pipeline configs
- Store pipeline configs in version control for reproducibility
- Prefix output filenames to avoid collisions with other workflows

In [ ]:
import shutil
from pathlib import Path

cleanup_files = [
    "../output/nb_pipeline_simple.json",
    "../output/nb_pipeline_complex.json",
    "../output/nb_pipe_people.json",
    "../output/nb_pipe_people.csv",
    "../output/nb_pipe_qa_raw.json",
    "../output/nb_pipe_qa_clean.json",
]
cleanup_dirs = [
    "../output/nb_pipe_qa_alpaca",
]

for f in cleanup_files:
    p = Path(f)
    if p.exists():
        p.unlink()
        print(f"Deleted: {f}")

for d in cleanup_dirs:
    p = Path(d)
    if p.exists():
        shutil.rmtree(p)
        print(f"Deleted: {d}")

print("\nCleanup complete.")